# Feature Engineer

En este notebook se realiza el feature engineer para crear o transformar nuevas variables que a priori sirvan para el entrenamiento de un modelo, para ello se pueden implementar diferentes tecnicas comunes y algunas custom.

Antes de realizar el feature engineer, se debe considerar lso aspectos mencionados en el EDA y según ello, diganisticar posibles técnicas que permitan mitigar esos aspectos, a modo resumen:

### Resumen de hallazgos del EDA (train)

Con base en el EDA sobre la partición de train,, se resumen a continuación los aspectos más relevantes detectados por variable/grupo de variables, el riesgo técnico que implican para el modelado, las soluciones propuestas para el feature engineering y su justificación desde la óptica del negocio (función de ganancia: +25% por transacción buena, ~100% de pérdida por fraude no detectado).

| Problema Identificado | Variables Afectadas | Riesgo Técnico | Soluciones Propuestas | Justificación de Negocio |
|---|---|---|---|---|
| Desbalanceo del target | `fraude` (~5% positivos) | El modelo tiende a favorecer la clase mayoritaria y a generar falsos negativos; accuracy es una métrica engañosa | Aprendizaje sensible al costo (class weights), umbral de decisión optimizado por ganancia esperada, evaluar con PR-AUC/curva de ganancia | Un fraude no detectado cuesta ~100% del monto vs. +25% de ganancia por transacción buena; el umbral debe maximizar la ganancia esperada, no la exactitud |
| Nulos moderados en numéricas | `b`, `c` (~8.7% nulos), `d`, `f`, `l`, `m` (<0.25% nulos) | Al exigir filas completas (p.ej. VIF) se descarta ~12.2% del train; imputar sin criterio puede sesgar el modelo si el nulo no es aleatorio | Missing indicator (flag binario de nulo) + imputación (mediana/constante) en vez de eliminar filas | El modelo debe poder scorear el 100% de las transacciones en producción, no solo el ~88% con datos completos |
| ALta cantidad de nulos | `o` (72.6% nulo) | Imputar o eliminar el nulo destruiría la señal más fuerte del dataset | Conservar el nulo como categoría explícita (no imputar); usar como feature binaria/categórica directamente | `o=N` tiene tasa de fraude 22.8% (lift 4.4x) vs. 2.1% cuando es nulo; es la variable más discriminante del dataset |
| Alta cardinalidad en categórica | `j` (7,558–8,324 categorías, patrón tipo ID) | One-hot encoding generaría sparsity extrema, overfitting y categorías nuevas no vistas en test/producción (cold start) | Target/frequency encoding, o feature binaria "categoría de alto riesgo" a partir de categorías con lift >10x | Categorías como `cat_eb8147e` (66.7% fraude) o `cat_440aeb1` (60.9% fraude) concentran señal explotable con soporte suficiente, sin necesidad de miles de dummies |
| Columnas tipo ID sin valor predictivo | `k` (100% único), `c` (90% único, revisar) | Usarlas como feature real favorece overfitting a un identificador que no generaliza a transacciones futuras | Excluir `k` del modelo (dejarla solo como llave); confirmar si `c` es un ID disfrazado o una variable continua legítima | Una decisión de aprobar/rechazar en producción no puede depender de un identificador que no existirá en transacciones futuras |
| Multicolinealidad moderada | `d`, `m` (corr. 0.59, VIF ≤1.61) | VIF y condition number en Regresión Logística (2.2) descartan multicolinealidad severa, pero `d` cambia de coeficiente al remover `m` (redundancia parcial) | Conservar ambas variables por ahora; monitorear el VIF si se agregan features derivadas de ellas en el futuro | Eliminar una variable sin evidencia fuerte de redundancia severa arriesga perder poder predictivo sin necesidad |
| Sesgo (skewness) extremo y outliers | `c`, `f`, `e`, `monto` (skew entre 6 y 142), `a`, `b` (sesgo alto) | Modelos lineales/logísticos son sensibles a la escala; los valores extremos pueden dominar el ajuste de coeficientes | Transformación log/Box-Cox o winsorización para modelos lineales; modelos basados en árboles (XGBoost/LightGBM) son robustos y pueden preferirse | Los outliers de `monto` suelen ser transacciones de alto valor, justo donde el costo de un fraude no detectado es mayor en términos absolutos — conviene tratarlos, no eliminarlos |
| Desbalance de categorías | `a`, `g` (Cramér's V despreciable, <0.06) | Agregar variables sin señal real añade ruido y dimensionalidad sin aportar poder predictivo | Evaluar exclusión o mantenerlas solo para interacciones; priorizar esfuerzo de feature engineering en `o`, `n`, `j` | Simplifica el modelo y reduce riesgo de overfitting sin sacrificar performance, enfocando el esfuerzo en las señales con impacto real |
| `score` como variable externa correlacionada con el target | `score` (corr. 0.17 con `fraude`) | Si proviene de un modelo/regla de negocio ya existente, usarla como feature puede enmascarar el problema real o generar dependencia de un sistema externo | Confirmar el origen de `score` antes de incluirla; si es un score de fraude preexistente, tratarla también como baseline a superar | Si ya existe un score de riesgo en producción, el nuevo modelo debe demostrar una mejora incremental en la ganancia esperada frente a ese score, no solo replicarlo |

De lo anterior, se peude definir una nueva lista de variables a considerar entre ellas podemos destacar varias tecnicas asociadas:

- Escalas logaritmicas
- Missing indicator
- Zero is flag
- Bins para monto
- Percentile Flag
- Codificacion ciclica para horas
- Agrupar categorías raras
- Aplicar PCA para d y m,o descartar alguna de las dos
- Features de agrupaciones 
- Ratios

# Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display
from pprint import pprint

ROOT_DIR = Path.cwd().parent if (Path.cwd().name == "02_feature_engineer") else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.feature_engineer_utils.preprocessing_features_laboratory import preprocessing

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


# Configs

In [2]:
DATASET_DIR = ROOT_DIR / "dataset"
TRAIN_PATH = DATASET_DIR / "train.csv"
OUTPUT_PATH = DATASET_DIR / "train_feat_engineer.csv"
CONFIG_PATH = ROOT_DIR / "src" / "configs" / "feature_engineer_config.yml"
TARGET_COL = "fraude"

TRAIN_PATH

PosixPath('/Users/dpiedrahita/proyectos/DS_pro/dataset/train.csv')

# Read Data

In [3]:
df = pd.read_csv(TRAIN_PATH, parse_dates=["fecha"])
print(f"Shape train: {df.shape}")
df.head()


Shape train: (104981, 19)


,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude
0,4,0.7388,6314.50,14.0,0.139279,24.0,BR,7,cat_381751d,0.937548,2361.0,442.0,1,NaN,Y,2020-03-08 00:02:15,22.18,25,0
1,4,0.7548,21171.09,20.0,0.514815,7.0,BR,2,cat_a024847,0.791998,2324.0,73.0,1,NaN,N,2020-03-08 00:04:25,6.00,7,0
2,4,0.9026,4012.83,50.0,0.274167,1.0,BR,3,cat_1d61c62,0.688592,235.0,232.0,1,N,Y,2020-03-08 00:08:23,26.67,91,1
3,4,0.8285,99612.95,1.0,0.000000,4.0,BR,28,cat_01a1725,0.654161,658.0,0.0,1,N,N,2020-03-08 00:08:39,40.01,91,1
4,2,0.5992,53526.36,2.0,0.000000,264.0,AR,34,cat_f1e7464,0.532994,2400.0,10.0,1,NaN,N,2020-03-08 00:09:01,6.25,93,0


## Configuración de las transformaciones

Cada técnica listada arriba se traduce en una o varias entradas del `config_list` que consume `preprocessing.apply_transformations`. Esta configuración vive en `src/configs/feature_engineer_config.yml` (no hardcodeada en el notebook), y se carga a continuación. La elección de columnas para cada técnica se apoya en los hallazgos de la tabla de EDA:

| Técnica | Columnas | Por qué |
|---|---|---|
| Escalas logarítmicas | `monto`, `c`, `f`, `e` | Skew alto (6 a 142), candidatas a transformación log |
| Missing indicator | `b`, `c`, `d`, `f`, `l`, `m` | Nulos moderados (<9%); `o` se excluye porque su nulo ya es la señal (se conserva como categoría, no se imputa) |
| Zero is flag | `e`, `f`, `m` | Mayor concentración real de ceros (44.3%, 16.7%, 10.6%); `b`/`h`/`l` tienen muy pocos ceros para ser informativos |
| Bins para monto | `monto` | Pedido explícito de la lista de técnicas |
| Percentile Flag | `monto` (p99) | Transacciones de alto valor, donde el costo de un fraude no detectado es mayor |
| Codificación cíclica para horas | `fecha` (sin y cos) | `fecha` tiene timestamp completo (hora:min:seg) |
| Agrupar categorías raras | `g` | 45 categorías, una domina con 74% (`BR`); candidata clásica a rare grouping |
| Features de agrupaciones | `j` (frequency + target encoding) | Alta cardinalidad (7,558+ categorías); one-hot es inviable, se usan encodings basados en agregados por categoría |
| PCA para `d` y `m` | `d`, `m` -> 1 componente | Corr. 0.59 entre ambas; se comprimen en un componente en vez de descartar una |
| Ratios | `monto` / `l` | Razón entre el monto de la transacción y `l` (variable numérica de mayor escala en el dataset) |

In [4]:
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config_list = yaml.safe_load(f)["config_list"]

print(f"Config cargada desde: {CONFIG_PATH}")

pprint(config_list)
print(len(config_list))


Config cargada desde: /Users/dpiedrahita/proyectos/DS_pro/src/configs/feature_engineer_config.yml
[{'column': 'a', 'new_column': 'a_freq_enc', 'process': 'frequency_encoding'},
 {'column': 'a', 'new_column': 'a_grouped', 'process': 'rare_grouping'},
 {'column': 'b', 'new_column': 'b_missing', 'process': 'missing_flag'},
 {'column': 'b', 'new_column': 'b_is_zero', 'process': 'zero_flag'},
 {'column': 'b', 'new_column': 'b_log', 'process': 'log_feature'},
 {'column': 'b',
  'new_column': 'b_top1pct',
  'params': {'quantile': 0.99},
  'process': 'top_percentile_flag'},
 {'column': 'b',
  'new_column': 'b_bin',
  'params': {'num_bins': 5},
  'process': 'quantile_bins_dropna'},
 {'column': 'c', 'new_column': 'c_missing', 'process': 'missing_flag'},
 {'column': 'c', 'new_column': 'c_log', 'process': 'log_feature'},
 {'column': 'c',
  'new_column': 'c_top1pct',
  'params': {'quantile': 0.99},
  'process': 'top_percentile_flag'},
 {'column': 'c',
  'new_column': 'c_bin',
  'params': {'num_bins

## Construcción de las features

In [5]:
prep = preprocessing(df)
features_df = prep.apply_transformations(config_list)
print(f"Shape features nuevas: {features_df.shape}")
features_df.head()

Shape features nuevas: (104981, 61)


,a_freq_enc,a_grouped,b_missing,b_is_zero,b_log,b_top1pct,b_bin,c_missing,c_log,c_top1pct,c_bin,d_missing,d_is_zero,d_log,d_top1pct,d_bin,e_is_zero,e_log,e_top1pct,e_bin,f_missing,f_is_zero,f_log,f_top1pct,f_bin,g_missing,g_freq_enc,g_grouped,h_is_zero,h_log,h_top1pct,h_bin,j_freq_enc,j_grouped,j_target_enc,l_missing,l_is_zero,l_log,l_top1pct,l_bin,m_missing,m_is_zero,m_log,m_top1pct,m_bin,o_freq_enc,o_target_enc,fecha_hour,fecha_weekday,fecha_is_weekend,fecha_hour_sin,fecha_hour_cos,monto_log,monto_top1pct,monto_bin,score_is_zero,score_log,score_top1pct,score_bin,pca_dm_1,ratio_monto_l
0,0.850821,4,0,0,0.553195,0,2,0,8.750762,0,1,0,0,2.708050,0,2,0,0.130395,0,0,0,0,3.218876,0,3,0,0.755489,BR,0,2.079442,0,1,0.004563,__OTHER__,0.042624,0,0,7.767264,0,2,0,0,6.093570,0,3,0.724388,0.051838,0,6,1,0.0,1.0,3.143290,0,2,0,3.258097,0,1,142.485292,0.009394
1,0.850821,4,0,0,0.562355,0,2,0,9.960439,0,1,0,0,3.044522,0,2,0,0.415293,0,2,0,0,2.079442,0,2,0,0.755489,BR,0,1.098612,0,0,0.000133,__OTHER__,0.023563,0,0,7.751475,0,2,0,0,4.304065,0,1,0.724388,0.051838,0,6,1,0.0,1.0,1.945910,0,0,0,2.079442,0,0,-226.040600,0.002582
2,0.850821,4,0,0,0.643221,0,4,0,8.297501,0,0,0,0,3.931826,1,3,0,0.242292,0,1,0,0,0.693147,0,0,0,0.755489,BR,0,1.386294,0,1,0.002648,__OTHER__,0.044569,0,0,5.463832,0,0,0,0,5.451038,0,2,0.113421,0.230557,0,6,1,0.0,1.0,3.320349,0,2,0,4.521789,0,4,-66.040159,0.113489
3,0.850821,4,0,0,0.603496,0,4,0,11.509057,0,3,0,0,0.693147,0,0,1,0.000000,0,0,0,0,1.609438,0,1,0,0.755489,BR,0,3.367296,0,4,0.000438,__OTHER__,0.119965,0,0,6.490724,0,0,0,1,0.000000,0,0,0.113421,0.228374,0,6,1,0.0,1.0,3.713816,0,3,0,4.521789,0,4,-299.693151,0.060805
4,0.101837,2,0,0,0.469504,0,0,0,10.887948,0,2,0,0,1.098612,0,0,1,0.000000,0,0,0,0,5.579730,0,4,0,0.204132,AR,0,3.555348,0,4,0.000076,__OTHER__,0.032399,0,0,7.783641,0,2,0,0,2.397895,0,0,0.724388,0.051838,0,6,1,0.0,1.0,1.981001,0,0,0,4.543295,0,4,-289.663017,0.002604


In [6]:
features_df.columns

Index(['a_freq_enc', 'a_grouped', 'b_missing', 'b_is_zero', 'b_log', 'b_top1pct', 'b_bin', 'c_missing', 'c_log',
       'c_top1pct', 'c_bin', 'd_missing', 'd_is_zero', 'd_log', 'd_top1pct', 'd_bin', 'e_is_zero', 'e_log',
       'e_top1pct', 'e_bin', 'f_missing', 'f_is_zero', 'f_log', 'f_top1pct', 'f_bin', 'g_missing', 'g_freq_enc',
       'g_grouped', 'h_is_zero', 'h_log', 'h_top1pct', 'h_bin', 'j_freq_enc', 'j_grouped', 'j_target_enc', 'l_missing',
       'l_is_zero', 'l_log', 'l_top1pct', 'l_bin', 'm_missing', 'm_is_zero', 'm_log', 'm_top1pct', 'm_bin',
       'o_freq_enc', 'o_target_enc', 'fecha_hour', 'fecha_weekday', 'fecha_is_weekend', 'fecha_hour_sin',
       'fecha_hour_cos', 'monto_log', 'monto_top1pct', 'monto_bin', 'score_is_zero', 'score_log', 'score_top1pct',
       'score_bin', 'pca_dm_1', 'ratio_monto_l'],
      dtype='str')

In [7]:
train_feat_engineer = pd.concat([df, features_df], axis=1)
print(f"Shape final (originales + nuevas): {train_feat_engineer.shape}")
train_feat_engineer.head()

Shape final (originales + nuevas): (104981, 80)


,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude,a_freq_enc,a_grouped,b_missing,b_is_zero,b_log,b_top1pct,b_bin,c_missing,c_log,c_top1pct,c_bin,d_missing,d_is_zero,d_log,d_top1pct,d_bin,e_is_zero,e_log,e_top1pct,e_bin,f_missing,f_is_zero,f_log,f_top1pct,f_bin,g_missing,g_freq_enc,g_grouped,h_is_zero,h_log,h_top1pct,h_bin,j_freq_enc,j_grouped,j_target_enc,l_missing,l_is_zero,l_log,l_top1pct,l_bin,m_missing,m_is_zero,m_log,m_top1pct,m_bin,o_freq_enc,o_target_enc,fecha_hour,fecha_weekday,fecha_is_weekend,fecha_hour_sin,fecha_hour_cos,monto_log,monto_top1pct,monto_bin,score_is_zero,score_log,score_top1pct,score_bin,pca_dm_1,ratio_monto_l
0,4,0.7388,6314.50,14.0,0.139279,24.0,BR,7,cat_381751d,0.937548,2361.0,442.0,1,NaN,Y,2020-03-08 00:02:15,22.18,25,0,0.850821,4,0,0,0.553195,0,2,0,8.750762,0,1,0,0,2.708050,0,2,0,0.130395,0,0,0,0,3.218876,0,3,0,0.755489,BR,0,2.079442,0,1,0.004563,__OTHER__,0.042624,0,0,7.767264,0,2,0,0,6.093570,0,3,0.724388,0.051838,0,6,1,0.0,1.0,3.143290,0,2,0,3.258097,0,1,142.485292,0.009394
1,4,0.7548,21171.09,20.0,0.514815,7.0,BR,2,cat_a024847,0.791998,2324.0,73.0,1,NaN,N,2020-03-08 00:04:25,6.00,7,0,0.850821,4,0,0,0.562355,0,2,0,9.960439,0,1,0,0,3.044522,0,2,0,0.415293,0,2,0,0,2.079442,0,2,0,0.755489,BR,0,1.098612,0,0,0.000133,__OTHER__,0.023563,0,0,7.751475,0,2,0,0,4.304065,0,1,0.724388,0.051838,0,6,1,0.0,1.0,1.945910,0,0,0,2.079442,0,0,-226.040600,0.002582
2,4,0.9026,4012.83,50.0,0.274167,1.0,BR,3,cat_1d61c62,0.688592,235.0,232.0,1,N,Y,2020-03-08 00:08:23,26.67,91,1,0.850821,4,0,0,0.643221,0,4,0,8.297501,0,0,0,0,3.931826,1,3,0,0.242292,0,1,0,0,0.693147,0,0,0,0.755489,BR,0,1.386294,0,1,0.002648,__OTHER__,0.044569,0,0,5.463832,0,0,0,0,5.451038,0,2,0.113421,0.230557,0,6,1,0.0,1.0,3.320349,0,2,0,4.521789,0,4,-66.040159,0.113489
3,4,0.8285,99612.95,1.0,0.000000,4.0,BR,28,cat_01a1725,0.654161,658.0,0.0,1,N,N,2020-03-08 00:08:39,40.01,91,1,0.850821,4,0,0,0.603496,0,4,0,11.509057,0,3,0,0,0.693147,0,0,1,0.000000,0,0,0,0,1.609438,0,1,0,0.755489,BR,0,3.367296,0,4,0.000438,__OTHER__,0.119965,0,0,6.490724,0,0,0,1,0.000000,0,0,0.113421,0.228374,0,6,1,0.0,1.0,3.713816,0,3,0,4.521789,0,4,-299.693151,0.060805
4,2,0.5992,53526.36,2.0,0.000000,264.0,AR,34,cat_f1e7464,0.532994,2400.0,10.0,1,NaN,N,2020-03-08 00:09:01,6.25,93,0,0.101837,2,0,0,0.469504,0,0,0,10.887948,0,2,0,0,1.098612,0,0,1,0.000000,0,0,0,0,5.579730,0,4,0,0.204132,AR,0,3.555348,0,4,0.000076,__OTHER__,0.032399,0,0,7.783641,0,2,0,0,2.397895,0,0,0.724388,0.051838,0,6,1,0.0,1.0,1.981001,0,0,0,4.543295,0,4,-289.663017,0.002604


**Notas:**
- `f_log` tendrá nulos adicionales en las filas donde `f <= -1` (tiene valores negativos hasta -4), ya que `log1p` requiere `x > -1`. Es un comportamiento documentado de `build_log_feature`, no un error.
- `pca_dm_1` es el único componente principal de `d`/`m` combinados; se puede usar como sustituto de ambas o en conjunto con ellas.
- Las columnas originales se conservan todas (incluida `k`, que sigue sin usarse como feature de modelado, solo como identificador).

# Guardar dataset con feature engineering

In [8]:
train_feat_engineer.to_csv(OUTPUT_PATH, index=False)
print(f"Guardado: {OUTPUT_PATH}")


Guardado: /Users/dpiedrahita/proyectos/DS_pro/dataset/train_feat_engineer.csv
